In [1]:
import litellm
from importlib.metadata import version
print("litellm:", version("litellm"))          # 1.74.15.post2
from crewai.llm import LITELLM_AVAILABLE
print("LITELLM_AVAILABLE:", LITELLM_AVAILABLE)  # True




litellm: 1.74.15.post2
LITELLM_AVAILABLE: True


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # Needs MOONSHOT_API_KEY and SERPAPI_API_KEY in .env

True

In [10]:
from crewai import LLM

llm = LLM(
    model='moonshot/kimi-k2.5',
    api_key=os.getenv('MOONSHOT_API_KEY')
)

llm.model

'moonshot/kimi-k2.5'

In [11]:
from crewai.tools import tool
from langchain_community.utilities import SerpAPIWrapper

serpapi = SerpAPIWrapper(serpapi_api_key=os.getenv('SERPAPI_API_KEY'))

@tool("Web Search")
def search(query: str) -> str:
    """Search the web for real-time information about current events, people, weather, sports, etc."""
    return serpapi.run(query)

In [5]:
search.run(query='what are the upcoming games for mlb today?')

{'title': 'MLB',
 'thumbnail': 'https://serpapi.com/searches/6a3efb221a29fed53a61c892/images/q-5ZviWn8N8S913GBQ6fAnSYHo29X_qkuUjVBSEzkr8.png',
 'games': [{'tournament': 'MLB',
   'venue': 'Comerica Park',
   'venue_kgmid': '/m/01wn2j',
   'date': 'Today',
   'time': '6:40 PM',
   'video_highlights': {'link': 'https://www.mlb.com/stories/game-preview/824255',
    'thumbnail': 'https://serpapi.com/searches/6a3efb221a29fed53a61c892/images/5URbzGOFLhe1joNY9viVcbg5aMH7yKrfRdViGWr8mG75b8VRwDgp6H_45wmGD7ZrqD-Fe4y1PgWgOQuBgFgFqg.jpeg'},
   'teams': [{'name': 'Astros', 'kgmid': '/m/03m1n'},
    {'name': 'Tigers', 'kgmid': '/m/02d02'}]},
  {'tournament': 'MLB',
   'venue': 'PNC Park',
   'venue_kgmid': '/m/03729q',
   'date': 'Today',
   'time': '6:40 PM',
   'video_highlights': {'link': 'https://www.mlb.com/stories/game-preview/823363',
    'thumbnail': 'https://serpapi.com/searches/6a3efb221a29fed53a61c892/images/5URbzGOFLhe1joNY9viVcbTynVKcEuAyKEahQPKdWGV-Qd14bZXfx_7Q0NeyLvhfXb_ObYhj3fjorppQs

In [12]:
from crewai import Agent

researcher = Agent(
    role='Senior Research Analyst',
    goal='Find comprehensive and accurate information about a given topic with a focus on recent developments',
    backstory='You are an experienced research analyst with a talent for finding relevant information and organizing it clearly.',
    tools=[search],
    llm=llm,
    verbose=True
)

writer = Agent(
    role='Content Writer',
    goal='Craft engaging, well-structured content based on research findings',
    backstory='You are a skilled writer with a passion for making complex topics accessible and engaging.',
    llm=llm,
    verbose=True
)

In [13]:
from crewai import Task

research_task = Task(
    description='Research the latest trends in the AI agent industry and provide a summary of the top 3 developments.',
    expected_output='A summary of the top 3 trending developments in the AI agent industry with a unique perspective on their significance.',
    agent=researcher
)

write_task = Task(
    description='Write an engaging 4-paragraph blog post about the AI agent industry based on the research findings.',
    expected_output='A 4-paragraph blog post in markdown format that is engaging, informative, and avoids complex jargon.',
    agent=writer,
    context=[research_task],
    output_file='blog-posts/new_post.md'
)

In [14]:
from crewai import Crew, Process

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=True
)

In [15]:
result = crew.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  48b5c995-a8c9-4da3-8b86-f1691a7a935d                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the latest trends in the AI agent industry and provide a summary of the top 3 developments.     │
│  ID: 5e131ebf-d4d4-4f46-b58b-8c290aa76706                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Research the latest trends in the AI agent industry and provide a summary of the top 3 developments.     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'latest trends AI agent industry 2024 developments'}                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool web_search executed with result: ['In 2024, AI agents are no longer a niche interest. Companies across industries are getting more serious about incorporating agents into their workflows - from ...', 'The AI agents market size was va...

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The AI world has crossed a threshold. We're no longer just chatting with clever chatbots— we're watching       │
│  fully autonomous agents tackle complex projects from start to finish without constant hand-holding. The        │
│  arrival of systems like Manus AI, alongside OpenAI's Operator and Devin AI, marks a fundamental shift from     │
│  narrow digital tools to general-purpose assistants that can research, code, debug, and build entire websites   │
│  on their own. These systems don't just answer questions; they independently break down ambitious goals, plan   │
│  their approach, and adapt when they hit roadblocks. Of course, this newfound freedom comes with growing        │
│  pains—these agents occasionally stumble on more complex tasks, reminding us that capability and reliability    │
│  are still finding their balance.                                                                               │
│                                                                                                                 │
│  But the real revolution isn't just about individual genius—it's about teamwork. The industry is rapidly        │
│  moving away from the idea of one AI doing everything, instead embracing ecosystems where specialized agents    │
│  collaborate under smart coordination. Think of it like a human project team: rather than a single generalist   │
│  trying to handle research, coding, and analysis alone, you have expert agents for each role working in         │
│  concert, managed by coordinating systems that keep everything on track. This multi-agent approach is           │
│  exploding in popularity, with the market projected to surge from $7 billion to over $375 billion in the next   │
│  decade. Companies are increasingly viewing these digital workforces not as simple automation tools, but as     │
│  capable teams that can manage everything from smart cities to global supply chains.                            │
│                                                                                                                 │
│  Perhaps most telling of the industry's maturation is seeing fierce rivals become collaborators. In a           │
│  surprising move, OpenAI and Anthropic co-founded the Agentic AI Foundation under the Linux Foundation to       │
│  create shared standards for how AI agents behave, communicate, and stay safe. It's the digital equivalent of   │
│  agreeing on universal charging ports—ensuring agents from different companies can work together seamlessly     │
│  rather than existing in isolated silos. This push for standardization, combined with new development           │
│  protocols and specialized tools like Claude Opus 4.5, signals that AI agents are transitioning from            │
│  experimental novelties to essential business infrastructure that enterprises can actually trust and deploy at  │
│  scale.                                                                                                         │
│                                                                                                                 │
│  Taken together, these developments paint a clear picture of where technology is heading. We're witnessing the  │
│  birth of a new computing paradigm where AI doesn't just respond to prompts—it autonomously accomplishes        │
│  objectives. Whether working solo on specialized tasks 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Write an engaging 4-paragraph blog post about the AI agent industry based on the research findings.            │
│  Agent:                                                                                                         │
│  Content Writer                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'llm_call_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  48b5c995-a8c9-4da3-8b86-f1691a7a935d                                                                           │
│  Final Output: The AI world has crossed a threshold. We're no longer just chatting with clever chatbots— we're  │
│  watching fully autonomous agents tackle complex projects from start to finish without constant hand-holding.   │
│  The arrival of systems like Manus AI, alongside OpenAI's Operator and Devin AI, marks a fundamental shift      │
│  from narrow digital tools to general-purpose assistants that can research, code, debug, and build entire       │
│  websites on their own. These systems don't just answer questions; they independently break down ambitious      │
│  goals, plan their approach, and adapt when they hit roadblocks. Of course, this newfound freedom comes with    │
│  growing pains—these agents occasionally stumble on more complex tasks, reminding us that capability and        │
│  reliability are still finding their balance.                                                                   │
│                                                                                                                 │
│  But the real revolution isn't just about individual genius—it's about teamwork. The industry is rapidly        │
│  moving away from the idea of one AI doing everything, instead embracing ecosystems where specialized agents    │
│  collaborate under smart coordination. Think of it like a human project team: rather than a single generalist   │
│  trying to handle research, coding, and analysis alone, you have expert agents for each role working in         │
│  concert, managed by coordinating systems that keep everything on track. This multi-agent approach is           │
│  exploding in popularity, with the market projected to surge from $7 billion to over $375 billion in the next   │
│  decade. Companies are increasingly viewing these digital workforces not as simple automation tools, but as     │
│  capable teams that can manage everything from smart cities to global supply chains.                            │
│                                                                                                                 │
│  Perhaps most telling of the industry's maturation is seeing fierce rivals become collaborators. In a           │
│  surprising move, OpenAI and Anthropic co-founded the Agentic AI Foundation under the Linux Foundation to       │
│  create shared standards for how AI agents behave, communicate, and stay safe. It's the digital equivalent of   │
│  agreeing on universal charging ports—ensuring agents from different companies can work together seamlessly     │
│  rather than existing in isolated silos. This push for standardization, combined with new development           │
│  protocols and specialized tools like Claude Opus 4.5, signals that AI agents are transitioning from            │
│  experimental novelties to essential business infrastructure that enterprises can actually trust and deploy at  │
│  scale.                                                                                                         │
│                                                                                                                 │
│  Taken together, these developments paint a clear pict

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [16]:
print(result.raw)

# The Silent Revolution: How AI Agents Are Rewiring the Modern Workplace

Something extraordinary is happening in offices around the world, and you might not even notice it. The AI agent industry—once a playground for experimental prototypes—is quietly maturing into the backbone of how modern businesses operate. This isn't just another tech trend; it's a fundamental rewiring of work itself, driven by three powerful forces that are transforming agents from clever toys into essential infrastructure.

First, the technology has become radically easier to deploy. When OpenAI, Microsoft, and Google released their enterprise platforms this year, they didn't just launch new products—they created a new foundation. Instead of building agents from scratch over months, companies can now assemble them like Lego blocks in days. These platforms handle the complex stuff—memory, decision-making frameworks, and coordination—so that non-technical teams can create specialized agents for finance, HR, or op

In [17]:
for i, task_output in enumerate(result.tasks_output):
    print(f'--- Task {i+1} ---')
    print(task_output.raw[:500])
    print()

--- Task 1 ---
The AI agent industry is experiencing a paradigm shift in 2024-2025, moving from experimental prototypes to enterprise-critical infrastructure. Here are the three most significant developments reshaping the landscape:

**1. Enterprise-Grade Agent Development Platforms Emerge as the New Operating System**

The release of OpenAI's Agents SDK and API, combined with Microsoft's evolution of Copilot into a full agent ecosystem and Google's Vertex AI Agentic Tools, represents more than just product la

--- Task 2 ---
# The Silent Revolution: How AI Agents Are Rewiring the Modern Workplace

Something extraordinary is happening in offices around the world, and you might not even notice it. The AI agent industry—once a playground for experimental prototypes—is quietly maturing into the backbone of how modern businesses operate. This isn't just another tech trend; it's a fundamental rewiring of work itself, driven by three powerful forces that are transforming agents from clever t

In [18]:
result.token_usage

UsageMetrics(total_tokens=9358, prompt_tokens=4872, cached_prompt_tokens=1024, completion_tokens=4486, reasoning_tokens=2282, cache_creation_tokens=0, successful_requests=8)

## 6. Structured Output

Use a Pydantic model to get typed, structured results from a task.

In [16]:
from pydantic import BaseModel
from typing import List


class TrendReport(BaseModel):
    topic: str
    trends: List[str]
    summary: str

In [17]:
structured_researcher = Agent(
    role='Trend Analyst',
    goal='Identify and summarize the top trends for a given topic',
    backstory='You are a data-driven analyst who excels at identifying patterns and summarizing them concisely.',
    tools=[search],
    llm=llm,
    verbose=True
)

structured_task = Task(
    description='Research the top 3 trends in large language models right now.',
    expected_output='A structured report with the topic, a list of 3 trends, and a brief overall summary.',
    agent=structured_researcher,
    output_pydantic=TrendReport
)

structured_crew = Crew(
    agents=[structured_researcher],
    tasks=[structured_task],
    process=Process.sequential,
    verbose=True
)

In [18]:
structured_result = structured_crew.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  a07c23ea-19bf-4d30-abad-3b69e12e31c5                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the top 3 trends in large language models right now.                                            │
│  ID: ae7a79d3-a511-4eee-b98c-5b9251f8a65e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Trend Analyst                                                                                           │
│                                                                                                                 │
│  Task: Research the top 3 trends in large language models right now.                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#6) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'top trends in large language models 2024'}                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool web_search executed with result: ['This paper provides a comprehensive survey to capture the progression of advances in language models.', 'The global large language models market size was estimated at USD 5617.4 million in 2024 and ...

...

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'llm_call_started' (expected 
'task_started')

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Research the top 3 trends in large language models right now.                                                  │
│  Agent:                                                                                                         │
│  Trend Analyst                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'llm_call_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  a07c23ea-19bf-4d30-abad-3b69e12e31c5                                                                           │
│  Final Output: {                                                                                                │
│    "topic": "Large Language Models (LLMs)",                                                                     │
│    "trends": [                                                                                                  │
│      "Reasoning-focused models with reinforcement learning: The emergence of reasoning-optimized LLMs that      │
│  spend more time thinking before responding, evolving from Chain-of-Thought prompting to product-level          │
│  implementations like OpenAI's o1 series, enabling enhanced problem-solving and complex task completion.",      │
│      "Smaller, more efficient models: A significant shift toward compact, lightweight models that maintain      │
│  high performance while reducing computational costs, memory demands, and inference latency - driven by the     │
│  need for practical deployment and edge computing capabilities.",                                               │
│      "Multimodal integration: The convergence of text, vision, and audio capabilities into unified models       │
│  (exemplified by GPT-4o and similar offerings), enabling real-time processing across multiple input types and   │
│  opening new applications in interactive AI systems."                                                           │
│    ],                                                                                                           │
│    "summary": "The LLM landscape in 2024-2025 is defined by a shift toward more intelligent, efficient, and     │
│  versatile models. Key developments include reasoning-focused architectures that emulate human-like thinking    │
│  processes, a push for smaller yet capable models to improve accessibility and deployment, and the rapid        │
│  expansion of multimodal capabilities allowing seamless interaction across text, images, and audio. These       │
│  trends collectively point toward more practical, cost-effective, and capable AI systems entering mainstream    │
│  use."                                                                                                          │
│  }                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [22]:
report = structured_result.pydantic

print(f'Topic: {report.topic}')
print(f'Summary: {report.summary}')
for i, trend in enumerate(report.trends, 1):
    print(f'  {i}. {trend}')

Topic: Large Language Models
Summary: The large language model landscape is rapidly evolving with three dominant trends: multimodal capabilities that integrate vision and language, explosive growth in open-source models challenging proprietary systems, and increasing focus on efficiency through model compression and edge deployment. These trends reflect the industry's push toward more capable, accessible, and practical AI systems.
  1. Multimodal Integration: LLMs are rapidly expanding beyond text to seamlessly process images, audio, and video, with models like GPT-4V, Gemini, and open-source alternatives demonstrating sophisticated vision-language understanding capabilities that enable new applications in document analysis, visual reasoning, and interactive AI assistants.
  2. Open-Source Competition: The open-source community is closing the gap with proprietary models, with releases like Llama 3, Mixtral, and DBRX offering near state-of-the-art performance while providing transparenc

## 7. Custom Tool Example

Any Python function can become a CrewAI tool with the `@tool` decorator.

In [19]:
@tool("Python Executor")
def execute_python(code: str) -> str:
    """Execute a Python expression and return the result."""
    try:
        return str(eval(code))
    except Exception as e:
        return f'Error: {e}'


math_agent = Agent(
    role='Math Assistant',
    goal='Solve math problems by executing Python code',
    backstory='You are a precise math assistant who always uses the Python Executor tool to compute answers.',
    tools=[execute_python],
    llm=llm,
    verbose=True
)

math_task = Task(
    description='What is the 20th Fibonacci number? Use the Python Executor tool to compute it.',
    expected_output='The 20th Fibonacci number as a single integer.',
    agent=math_agent
)

math_crew = Crew(
    agents=[math_agent],
    tasks=[math_task],
    process=Process.sequential,
    verbose=True
)

In [20]:
math_result = math_crew.kickoff()
print(math_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  8348e4f6-54ef-4237-b10c-223f4fa3600f                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: What is the 20th Fibonacci number? Use the Python Executor tool to compute it.                           │
│  ID: a35b504f-ceaa-45a6-8101-e7316efa18be                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Assistant                                                                                          │
│                                                                                                                 │
│  Task: What is the 20th Fibonacci number? Use the Python Executor tool to compute it.                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool python_executor executed with result: Error: invalid syntax (<string>, line 2)...

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: python_executor                                                                                          │
│  Args: {'code': '\ndef fibonacci(n):\n    """Calculate the nth Fibonacci number (0-indexed: F(0)=0,             │
│  F(1)=1)"""\n    if n == 0:\n        return 0\n    elif n == 1:\n        return 1\n    else:\n        a, b      │
│  =...                                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  What is the 20th Fibonacci number? Use the Python Executor tool to compute it.                                 │
│  Agent:                                                                                                         │
│  Math Assistant                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'llm_call_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  8348e4f6-54ef-4237-b10c-223f4fa3600f                                                                           │
│  Final Output: 6765                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

6765


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯